In [1]:
import os
import shutil
import pandas as pd
import subprocess

import sys
sys.path.append('..')
from utils.audio_util import convert_wav_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy

In [2]:
DEST_DIR = "../data/converted/LjSpeech-to-vctk"
DEST_TEXT_PATH = os.path.join(DEST_DIR, "txt/LjSpeech")
DEST_AUDIO_PATH = os.path.join(DEST_DIR, "wav44/LjSpeech")
SRC_PATH = '../data/raw/LjSpeech'
# Clean and create directories
if os.path.exists(DEST_DIR):
   print("Clearing destination folder")
   shutil.rmtree(DEST_DIR)
os.makedirs(DEST_TEXT_PATH, exist_ok=True)
os.makedirs(DEST_AUDIO_PATH, exist_ok=True)

text_file = [f for f in os.listdir(SRC_PATH) if f.endswith(".csv")]

file_path = os.path.join(SRC_PATH, text_file[0])

if os.path.exists(file_path):  # Check if file exists
    df = pd.read_csv(file_path, sep="|", header=None, names=["ID", "Transcription", "Normalized Transcription"])
    print(df.head())
else:
    print(f"Error: File not found at {file_path}")




Clearing destination folder
           ID                                      Transcription  \
0  LJ001-0001  Printing, in the only sense with which we are ...   
1  LJ001-0002                     in being comparatively modern.   
2  LJ001-0003  For although the Chinese took impressions from...   
3  LJ001-0004  produced the block books, which were the immed...   
4  LJ001-0005  the invention of movable metal letters in the ...   

                            Normalized Transcription  
0  Printing, in the only sense with which we are ...  
1                     in being comparatively modern.  
2  For although the Chinese took impressions from...  
3  produced the block books, which were the immed...  
4  the invention of movable metal letters in the ...  


In [3]:
df

,ID,Transcription,Normalized Transcription
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...
...,...,...,...
13095,LJ050-0274,made certain recommendations which it believes...,made certain recommendations which it believes...
13096,LJ050-0275,materially improve upon the procedures in effe...,materially improve upon the procedures in effe...
13097,LJ050-0276,"As has been pointed out, the Commission has no...","As has been pointed out, the Commission has no..."
13098,LJ050-0277,with the active cooperation of the responsible...,with the active cooperation of the responsible...


In [4]:
mismatched_rows = df[df["Transcription"] != df["Normalized Transcription"]]

if mismatched_rows.empty:
    print("All is OK ✅")
else:
    print("Mismatched rows found:")
    print(mismatched_rows)

Mismatched rows found:
               ID                                      Transcription  \
6      LJ001-0007  the earliest book printed with movable types, ...   
23     LJ001-0024  But the first Bible actually dated (which also...   
30     LJ001-0031  In 1465 Sweynheim and Pannartz began printing ...   
33     LJ001-0034  They printed very few books in this type, thre...   
37     LJ001-0038  while in 1470 at Paris Udalric Gering and his ...   
...           ...                                                ...   
13057  LJ050-0236  Thus, in the 4 months following the assassinat...   
13058  LJ050-0237  the FBI, on 16 separate occasions, supplied a ...   
13060  LJ050-0239            From February 11 through June 30, 1964,   
13061  LJ050-0240  the Service had the advantage of 9,500 hours o...   
13087  LJ050-0266     The exchange of letters dated August 31, 1964,   

                                Normalized Transcription  
6      the earliest book printed with movable types, 

In [5]:
SRC_WAV_PATH = os.path.join(SRC_PATH, 'wavs')
count = 1

if os.path.exists(SRC_WAV_PATH):
    list_wavs = os.listdir(SRC_WAV_PATH)
    for w in list_wavs:
        txt_name = (w.split('.'))[0]
        src_wav_file = os.path.join(SRC_WAV_PATH, w)
        dest_wav_file = os.path.join(DEST_AUDIO_PATH, f"LjSpeech_{count:03d}.flac")
        if os.path.exists(src_wav_file) and txt_name in df["ID"].values:
            original_text = df.loc[df["ID"] == txt_name, "Transcription"].values[0]
            dest_txt_file = os.path.join(DEST_TEXT_PATH, f"LjSpeech_{count:03d}.txt")
            
            with open(dest_txt_file, "w", encoding="utf-8") as f:
                f.write(original_text)
            
            convert_wav_to_flac(src_wav_file,dest_wav_file)

            count += 1
        else:
            print(f"Error: Source file not found at {src_wav_file} or {txt_name} don't exist")
else:
    print(f"Error: File not found at {file_path}")

# Resample, trim, and normalize audio

In [6]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/LjSpeech-to-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav32 to wav16_silence_trimmed
src_dir = "../data/converted/LjSpeech-to-vctk/wav44"
dst_dir = "../data/converted/LjSpeech-to-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [7]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 4

resample_audios(
  input_folders="../data/converted/LjSpeech-to-vctk/wav16_silence_trimmed",
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 13100 files...


100%|██████████| 13100/13100 [00:30<00:00, 428.33it/s]

Done !


In [8]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder="../data/converted/LjSpeech-to-vctk/wav16_silence_trimmed",
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /Users/natyaninchayanuwong/.cache/torch/hub/master.zip


Found 13100 .flac files to process


Processing files: 100%|██████████| 13100/13100 [07:17<00:00, 29.96it/s]


Processing complete


In [9]:
normalize_audio_files(
    input_dir='../data/converted/LjSpeech-to-vctk/wav16_silence_trimmed',
    )

Normalizing audio files: 100%|██████████| 13100/13100 [1:14:09<00:00,  2.94it/s]
